<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/3%20-%20Regress%C3%A3o%20Log%C3%ADstica/3.2_regressao_softmax_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 3.2 — Regressão softmax no MNIST

## Objetivos

- Aplicar a mesma regressão softmax do notebook 3.1 ao reconhecimento de dígitos manuscritos (MNIST).
- Transformar cada imagem 28×28 num vetor de 784 atributos com `view(-1, 28*28)`.
- Treinar `nn.Linear(784, 10)` com 1000 amostras e acompanhar custo e acurácia no treino e na validação.
- Interpretar os pesos de cada classe como imagens 28×28.

## O problema

Dada a imagem de um dígito escrito à mão, qual é o dígito (0 a 9)?

- **Entrada**: uma imagem em tons de cinza de 28 × 28 pixels, ou seja, 784 atributos (no 3.1 eram 2).
- **Saída esperada**: o índice da classe correta, de 0 a 9 (no 3.1 eram 3 classes).
- **Modelo**: o mesmo do 3.1, uma camada linear com uma saída (logit) por classe: $z = Wx + b$, com $W$ de shape (10, 784) e $b$ de shape (10,).
- **Função de custo**: a mesma cross-entropy, `nn.CrossEntropyLoss`, que recebe os logits.

O MNIST tem 60 mil imagens de treino e 10 mil de teste. Aqui usamos só 1000 imagens de treino, todas de uma vez em cada época (batch completo). O notebook 3.4 usa as 60 mil com mini-batches. Como o modelo é linear, cada classe tem um vetor de 784 pesos que pode ser visto como uma imagem 28 × 28: no final vamos interpretar o que o modelo aprendeu.

## Importação das bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
from torchvision.datasets import MNIST

torch.manual_seed(0)
np.random.seed(0)

## Dados: carregamento do MNIST

`torchvision.datasets.MNIST` baixa os arquivos para `./data` na primeira execução (o mesmo diretório do notebook 3.4) e os reutiliza depois. Os atributos `.data` e `.targets` do dataset contêm todas as imagens (tensor `uint8`) e todos os rótulos. Eles não têm relação com o antigo `tensor.data` do PyTorch, que não usamos mais.

In [ ]:
mnist_train = MNIST(root='./data', train=True, download=True)
mnist_test = MNIST(root='./data', train=False, download=True)

x_train, y_train = mnist_train.data, mnist_train.targets

print("Amostras de treinamento:", x_train.size(0))

print("\nDimensões dos dados das imagens:   ", x_train.size())
print("Valores mínimo e máximo dos pixels:", torch.min(x_train), torch.max(x_train))
print("Tipo dos dados das imagens:        ", type(x_train), x_train.dtype)
print("Tipo das classes das imagens:      ", type(y_train), y_train.dtype)

### Normalização e seleção dos dados de treinamento

Os pixels vão de 0 a 255 (`uint8`). Convertemos para `float32` e dividimos por 255, para que fiquem entre 0 e 1. Neste exemplo utilizaremos apenas as 1000 primeiras amostras de treinamento.

In [ ]:
x_train = x_train.float() / 255.

n_samples_train = 1000
x_train = x_train[:n_samples_train]
y_train = y_train[:n_samples_train]

print("Amostras de treinamento:", x_train.size(0))

print("\nDimensões dos dados das imagens:   ", x_train.size())
print("Valores mínimo e máximo dos pixels:", torch.min(x_train), torch.max(x_train))
print("Tipo dos dados das imagens:        ", x_train.dtype)
print("Tipo das classes das imagens:      ", y_train.dtype)

### Normalização e seleção dos dados de validação

Para medir o desempenho em imagens que o modelo não usou no treinamento, separamos 1000 imagens do conjunto de teste do MNIST como conjunto de validação, com a mesma normalização.

In [ ]:
x_val, y_val = mnist_test.data, mnist_test.targets
x_val = x_val.float() / 255.

n_samples_val = 1000
x_val = x_val[:n_samples_val]
y_val = y_val[:n_samples_val]

print("Amostras de validação:", x_val.size(0))
print("Dimensões dos dados das imagens de validação:", x_val.size())
print("Distribuição das classes no treino:   ", torch.bincount(y_train).tolist())
print("Distribuição das classes na validação:", torch.bincount(y_val).tolist())

### Visualizando os dados

In [ ]:
n_samples = 24

# make_grid espera (N, canais, altura, largura): unsqueeze(dim=1) acrescenta o canal → (24, 1, 28, 28)
grid = torchvision.utils.make_grid(x_train[:n_samples].unsqueeze(dim=1), pad_value=1.0, padding=1, nrow=12)

plt.figure(figsize=(15, 3))
plt.imshow(grid.numpy().transpose(1, 2, 0))
plt.axis('off')
plt.show()
print('classes:', y_train[:n_samples].view(2, 12).tolist())

### Visualizando uma imagem com o matplotlib

In [ ]:
image  = x_train[0]
target = y_train[0]

plt.imshow(image.numpy(), cmap='gray')
plt.title(f'classe: {target.item()}')
plt.show()

## Modelo

In [ ]:
model = nn.Linear(28*28, 10)  # 28*28 = 784 atributos de entrada e 10 neurônios na saída (um por dígito)

### Testando uma predição com poucas amostras

`nn.Linear` espera cada amostra como um vetor. `x.view(-1, 28*28)` muda o shape de (5, 28, 28) para (5, 784) sem copiar os dados; o `-1` deixa o PyTorch calcular essa dimensão (aqui, 5 amostras).

In [ ]:
xin = x_train[:5].view(-1, 28*28)
print('xin:', xin.shape)
score = model(xin)  # logits, shape (5, 10)
score

Os logits não são probabilidades. A softmax os transforma em probabilidades que somam 1 em cada linha; com os pesos iniciais aleatórios, todas as classes ficam perto de 0,1.

In [ ]:
probs = F.softmax(score, dim=1)
print(probs.detach().numpy().round(3))  # .detach(): tensor sem histórico de gradientes, para converter em NumPy
print('soma de cada linha:', probs.sum(dim=1).detach())

## Treinamento

### Inicialização dos hiperparâmetros, da função de custo e do otimizador

In [ ]:
n_epochs = 100
lr = 0.5

# CrossEntropyLoss como função de custo: recebe os logits (sem softmax) e os índices das classes
criterion = nn.CrossEntropyLoss()

# Gradiente descendente
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

### Laço de treinamento com validação

Em cada época:

1. **Treino**: forward com as 1000 imagens, custo, `zero_grad()`, `backward()` e `step()`. Com batch completo, uma época é uma única iteração.
2. **Validação**: forward com as imagens de validação dentro de `torch.no_grad()`. Medimos custo e acurácia, mas não calculamos gradientes nem atualizamos parâmetros.

A acurácia de treino é medida com as saídas calculadas antes do `step()` daquela época.

In [ ]:
x_train_flat = x_train.view(-1, 28 * 28)  # (1000, 784)
x_val_flat = x_val.view(-1, 28 * 28)      # (1000, 784)

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for i in range(n_epochs):
    # Treino
    outputs = model(x_train_flat)       # logits (1000, 10)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())
    _, predicts = torch.max(outputs, 1)
    train_accuracies.append((predicts == y_train).float().mean().item())

    # Validação: sem gradientes e sem atualizar parâmetros
    with torch.no_grad():
        outputs_val = model(x_val_flat)
        loss_val = criterion(outputs_val, y_val)
        val_losses.append(loss_val.item())

        _, predicts_val = torch.max(outputs_val, 1)
        val_accuracies.append((predicts_val == y_val).float().mean().item())

print(f'Época {n_epochs}: custo treino {train_losses[-1]:.4f}, acurácia treino {train_accuracies[-1]:.4f}, '
      f'custo validação {val_losses[-1]:.4f}, acurácia validação {val_accuracies[-1]:.4f}')

### Visualizando custo e acurácia durante o treinamento

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='treino')
plt.plot(val_losses, label='validação')
plt.title('Custo ao longo das épocas')
plt.xlabel('época')
plt.ylabel('custo (cross-entropy)')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='treino')
plt.plot(val_accuracies, label='validação')
plt.title('Acurácia ao longo das épocas')
plt.xlabel('época')
plt.ylabel('acurácia')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

O custo e a acurácia de validação ficam piores que os de treino: o modelo foi ajustado às 1000 imagens de treino e é avaliado em imagens que não viu. Com poucas amostras para 7.850 parâmetros, essa diferença é esperada; o notebook 4.3 trata da generalização e do sobreajuste (overfitting). Repare também que as curvas de custo continuam descendo quando as de acurácia já quase não mudam.

## Avaliação

### Acurácia nos conjuntos de treinamento e de validação

Para avaliar usamos `model.eval()` e `torch.no_grad()`. Para `nn.Linear`, `model.eval()` não muda nada; é o padrão que usaremos sempre que avaliarmos um modelo.

In [ ]:
def predict(model, input_data):
    with torch.no_grad():
        outputs = model(input_data)
    _, predicts = torch.max(outputs, 1)
    return predicts

model.eval()
y_pred = predict(model, x_train_flat)
y_pred_val = predict(model, x_val_flat)
print(f'Acurácia no treino:    {(y_pred == y_train).float().mean().item():.4f}')
print(f'Acurácia na validação: {(y_pred_val == y_val).float().mean().item():.4f}')

### Matriz de confusão com dados de treinamento e de validação

Linhas: classe prevista; colunas: classe real.

In [ ]:
print('Matriz de confusão — treino:')
display(pd.crosstab(pd.Series(y_pred.numpy(), name='previsto'), pd.Series(y_train.numpy(), name='real')))
print('Matriz de confusão — validação:')
display(pd.crosstab(pd.Series(y_pred_val.numpy(), name='previsto'), pd.Series(y_val.numpy(), name='real')))

### Do logit à classe prevista, para uma imagem de validação

Para uma imagem, os 10 logits saem do modelo; a softmax os transforma em probabilidades; a classe prevista é a de maior valor (o mesmo índice nos dois casos).

In [ ]:
k = 0
with torch.no_grad():
    z = model(x_val_flat[k:k+1])[0]  # 10 logits
p = F.softmax(z, dim=0)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 3.2))
ax1.imshow(x_val[k].numpy(), cmap='gray'); ax1.set_title(f'classe correta: {y_val[k].item()}'); ax1.axis('off')
ax2.bar(range(10), z.numpy()); ax2.set_xticks(range(10)); ax2.set_title('logits z'); ax2.set_xlabel('classe')
ax3.bar(range(10), p.numpy(), color='g'); ax3.set_xticks(range(10)); ax3.set_ylim(0, 1)
ax3.set_title('probabilidades softmax(z)'); ax3.set_xlabel('classe')
plt.tight_layout()
plt.show()
print('classe prevista:', z.argmax().item(), ' probabilidade:', round(p.max().item(), 3))

## Visualizando a matriz de pesos treinados

A linha $k$ de $W$ tem 784 pesos, um por pixel, e o logit da classe $k$ é $z_k = \sum_j W_{k,j}\, x_j + b_k$. Reorganizada como imagem 28 × 28, a linha mostra quais pixels aumentam o logit da classe $k$ quando têm tinta (pesos positivos, claros) e quais o diminuem (pesos negativos, escuros).

In [ ]:
weights = model.state_dict()['weight']
print('weights:', weights.shape)

bias = model.state_dict()['bias']
print('bias:   ', bias.shape)

# Visualizando os pesos da classe 3
plt.imshow(weights[3, :].numpy().reshape((28, 28)), cmap='gray')
plt.title('pesos da classe 3')
plt.colorbar()
plt.show()

### Visualizando os pesos de todas as classes

In [ ]:
# cria um grid com as imagens; normalize=True ajusta cada imagem para a faixa [0, 1]
grid = torchvision.utils.make_grid(weights.view(-1, 1, 28, 28), normalize=True, scale_each=True,
                                   pad_value=1.0, padding=1, nrow=10)

plt.figure(figsize=(15, 2.5))
plt.imshow(grid.numpy().transpose(1, 2, 0))
plt.title('pesos das classes 0 a 9')
plt.axis('off')
plt.show()

### Diagrama da regressão softmax com visualização dos pesos W

![Diagrama: imagem de um dígito, os pesos de cada classe como imagem, os logits z0 a z9, softmax e argmax](https://raw.githubusercontent.com/vcasadei/images/master/RegressaoSoftmaxArgmaxNMIST.png)

## Exercícios

1. Escreva um laço de treinamento que, no final de cada época, imprima o número da época, o custo e a acurácia de treino. Use um modelo novo, para não continuar o treinamento do modelo acima.

In [ ]:
torch.manual_seed(0)
model_ex = nn.Linear(28*28, 10)
optimizer_ex = torch.optim.SGD(model_ex.parameters(), lr=lr)

for i in range(n_epochs):
    outputs = model_ex(x_train_flat)
    loss = criterion(outputs, y_train)
    optimizer_ex.zero_grad()
    loss.backward()
    optimizer_ex.step()

    # TODO: escreva sua solução aqui


2. Inclua no laço de treinamento a visualização dos gradientes referentes à classe do dígito 4 (`model.weight.grad[4]`, um vetor de 784 valores, visto como imagem 28 × 28) no final das épocas 1, 2, 5, 10, 50 e 100. Use outra vez um modelo novo. O que os gradientes mostram e como eles mudam ao longo do treinamento?

In [ ]:
torch.manual_seed(0)
model_ex = nn.Linear(28*28, 10)
optimizer_ex = torch.optim.SGD(model_ex.parameters(), lr=lr)
epochs_to_show = [1, 2, 5, 10, 50, 100]

# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

### Perguntas

1. Qual é o shape da matriz de entrada da rede no treinamento?
2. Qual é o shape da saída da rede?
3. Qual é o número total de parâmetros da rede, incluindo o bias?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

## Aprendizados

- A regressão softmax do Iris funciona sem mudanças para imagens: basta achatar cada imagem 28 × 28 num vetor de 784 atributos com `view(-1, 28*28)`.
- `nn.Linear(784, 10)` tem 7.850 parâmetros; cada linha de $W$ pode ser vista como uma imagem que indica quais pixels favorecem cada dígito.
- O gradiente da cross-entropy em relação aos logits é $p - \text{one-hot}(y)$: ele empurra a probabilidade da classe correta para 1 e a das outras para 0.
- O desempenho em validação é pior que no treino; acurácia de treino alta não garante acerto em imagens novas.
- A softmax que escolhe 1 entre 10 dígitos é a mesma que, num LLM, escolhe o próximo token entre dezenas de milhares de tokens do vocabulário.